In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit.transpiler import Target, CouplingMap
from qiskit.quantum_info import Operator
from qiskit_device_benchmarking.bench_code.mrb import MirrorRB, QuantumAwesomeness
import pickle
import time

# Define parameters for the simulated backend
num_qubits = 36
basis_gates = ["id", "h", "x", "y", "z", "s", "cx"]
p1 = 0.001  # 1-qubit gate error probability
p2 = 0.01  # 2-qubit gate error probability
rz_angle = np.pi / 2  # Match initial_entangling_angle

# Create a coupling map as a list of tuples
coupling_list = [(i, j) for i in range(num_qubits) for j in range(num_qubits) if i != j]

# Create a Target object and add gate definitions
coupling_map = CouplingMap(couplinglist=coupling_list)

# Create a Target object to define the gates, including rz explicitly
target = Target.from_configuration(
    num_qubits=num_qubits,
    basis_gates=basis_gates,
    coupling_map=coupling_map,
    custom_name_mapping={
        "id": Operator(np.array([[1, 0], [0, 1]])),  # Identity gate
        "h": Operator(np.array([[1, 1], [1, -1]]) / np.sqrt(2)),  # Hadamard gate
        "x": Operator(np.array([[0, 1], [1, 0]])),  # Pauli X gate
        "y": Operator(np.array([[0, -1j], [1j, 0]])),  # Pauli Y gate
        "z": Operator(np.array([[1, 0], [0, -1]])),  # Pauli Z gate
       #"rz": Operator(np.array([[np.exp(-1j * rz_angle / 2), 0], [0, np.exp(1j * rz_angle / 2)]])),  # RZ gate for pi/2
         "s": Operator(np.array([[1, 0], [0, 1j]])),  # S gate
        "cx": Operator(np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]]))  # CNOT gate
    }
)



# Create a noise model to emulate the NoisyBackend
noise_model = NoiseModel()

# Add depolarizing errors for 1-qubit and 2-qubit gates
error_1q = depolarizing_error(p1, 1)
error_2q = depolarizing_error(p2, 2)

# Apply errors to all basis gates except 'delay' and 'reset'
for gate in basis_gates:
    if gate in ["id", "h", "x", "y", "z", "s"]:
        noise_model.add_all_qubit_quantum_error(error_1q, gate)
    elif gate == "cx":
        noise_model.add_all_qubit_quantum_error(error_2q, gate)

# Set up the AerSimulator with stabilizer method, target, and noise model
backend = AerSimulator(
    method="stabilizer",
    noise_model=noise_model if (p1 > 0 or p2 > 0) else None,
    target=target,
    max_parallel_threads=0,  # Use all available threads
    max_parallel_experiments=0
)

# Number of shots per circuit
shots = 10000

# Reduced lengths and samples for faster debugging
lengths = [2, 4,10,20]
num_samples = 10

# Set up the experiment object
exp = MirrorRB(
    range(num_qubits),
    lengths=lengths,
    backend=backend,
    two_qubit_gate_density=0.25,
    num_samples=num_samples,
    initial_entangling_angle=np.pi/2,  # Clifford-compatible angle
)

# Set run options
exp.set_run_options(shots=shots)

# Run the experiment with timing
print("Starting simulation...")
start_time = time.time()
rb_data = exp.run()
end_time = time.time()
print(f"Simulation completed in {end_time - start_time:.2f} seconds")
print("Job IDs:", rb_data.job_ids)



In [ ]:
print('Pairs:', exp._pairs[0])
exp._static_trans_circuits[0].draw(fold=-1)

In [ ]:
if exp.analysis is not None:
	exp.analysis.set_options(analyzed_quantity='Effective Polarization')
	#exp.analysis.set_options(analyzed_quantity='Mutual Information')
	analysis = exp.analysis.run(rb_data)
else:
	print("Error: exp.analysis is None. Cannot set options or run analysis.")

In [ ]:
print(rb_data.status())
print(rb_data.figure_names)


In [ ]:
%matplotlib inline
# Debug: Inspect rb_data contents
print("Experiment data contents:")
try:
    data_entries = rb_data.data()
    for i, data in enumerate(data_entries):
        print(f"Data entry {i}: {data}")
        # Check required metadata
        required_keys = ["pairs", "singles", "target", "coupling_map"]
        metadata = data.get("metadata", {})
        for key in required_keys:
            print(f"  Metadata '{key}' present: {key in metadata}")
except Exception as e:
    print(f"Error accessing rb_data.data(): {e}")

# Save rb_data to a file for inspection
try:
    with open("rb_data.pkl", "wb") as f:
        pickle.dump(rb_data, f)
    print("Saved rb_data to rb_data.pkl for further inspection")
except Exception as e:
    print(f"Error saving rb_data: {e}")

# Try analysis with "Effective Polarization"
try:
    if exp.analysis is not None:
        exp.analysis.set_options(analyzed_quantity="Effective Polarization")
        print("Running analysis for Effective Polarization...")
        start_time = time.time()
        analysis = exp.analysis.run(rb_data)
        end_time = time.time()
        print(f"Analysis for Effective Polarization completed in {end_time - start_time:.2f} seconds")
        print("Analysis results for Effective Polarization:", analysis.analysis_results(dataframe=True))
        print("Available figures for Effective Polarization:", analysis.figure_names)

        # Try to display the figure
        try:
            fig = analysis.figure(0)
            display(fig)
        except Exception as e:
            print(f"Error accessing figure for Effective Polarization: {e}")
    else:
        print("Error: exp.analysis is None. Cannot set options or run analysis.")
except Exception as e:
    print(f"Error during analysis for Effective Polarization: {e}")


In [ ]:
analysis.figure(0)

In [ ]:
# Try analysis with "Mutual Information" as a fallback
try:
    if exp.analysis is not None:
        exp.analysis.set_options(analyzed_quantity="Mutual Information")
        print("Running analysis for Mutual Information...")
        start_time = time.time()
        analysis_mi = exp.analysis.run(rb_data)
        end_time = time.time()
        print(f"Analysis for Mutual Information completed in {end_time - start_time:.2f} seconds")
        mi_results_df = analysis_mi.analysis_results(dataframe=True)
        print("Analysis results for Mutual Information:\n", mi_results_df)
        print("Available figures for Mutual Information:", analysis_mi.figure_names)

        # Try to display the figure
        try:
            if analysis_mi.figure_names:
                fig = analysis_mi.figure(0)
                import matplotlib.pyplot as plt
                plt.show()
            else:
                print("No figures available for Mutual Information.")
        except Exception as e:
            print(f"Error accessing figure for Mutual Information: {e}")
    else:
        print("Error: exp.analysis is None. Cannot set options or run analysis.")
except Exception as e:
    print(f"Error during analysis for Mutual Information: {e}")


# Custom plotting based on original intent using QuantumAwesomeness
try:
    import matplotlib.pyplot as plt
    import numpy as np

    # Extract lengths from rb_data (using xval or index as fallback)
    data_entries = rb_data.data()
    lengths = [entry['metadata'].get('xval', i) for i, entry in enumerate(data_entries)]
    unique_lengths = sorted(list(set(lengths)))  # Unique lengths: [2, 4, 10]
    print("Derived unique lengths:", unique_lengths)

    # Initialize data structures
    num_qubits = len(exp.physical_qubits)  # Assuming 6 qubits
    ys = [[[] for _ in range(len(unique_lengths))] for _ in range(2)]  # 2 types x unique lengths
    yerrs = [[], []]

    # Compute mutual information using QuantumAwesomeness
    qa = QuantumAwesomeness(exp.backend.coupling_map)
    mmi = qa.mean_mutual_info(data_entries, [entry['metadata']['pairs'] for entry in data_entries])
    print("Mean mutual info:", mmi)

    # Aggregate mmi values by unique lengths
    for i, entry in enumerate(data_entries):
        length_idx = unique_lengths.index(entry['metadata'].get('xval', i))
        if mmi['paired'][i] is not np.nan:
            ys[0][length_idx].append(mmi['paired'][i])
        if mmi['single'][i] is not np.nan:
            ys[1][length_idx].append(mmi['single'][i])

    # Compute means and standard deviations
    for p in range(2):
        for j in range(len(unique_lengths)):
            if ys[p][j]:
                yerrs[p].append(np.std(ys[p][j]))
                ys[p][j] = np.mean(ys[p][j])
            else:
                yerrs[p].append(0)
                ys[p][j] = 0

    # Plot with error bars
    plt.errorbar(unique_lengths, ys[0], yerr=yerrs[0], fmt='o-', label='paired', capsize=5)
    plt.errorbar(unique_lengths, ys[1], yerr=yerrs[1], fmt='o-', label='singles', capsize=5)
    plt.yscale('log')
    plt.xlabel('Circuit Length (xval)')
    plt.ylabel('Mean Mutual Information')
    plt.title('Mutual Information vs Circuit Length')
    plt.legend()
    plt.show()

except Exception as e:
    print(f"Error during custom plotting: {e}")

In [ ]:
# Try analysis with "Mutual Information" as a fallback
try:
    if exp.analysis is not None:
        exp.analysis.set_options(analyzed_quantity="Mutual Information")
        print("Running analysis for Mutual Information...")
        start_time = time.time()
        analysis_mi = exp.analysis.run(rb_data)
        end_time = time.time()
        print(f"Analysis for Mutual Information completed in {end_time - start_time:.2f} seconds")
        mi_results_df = analysis_mi.analysis_results(dataframe=True)
        print("Analysis results for Mutual Information:\n", mi_results_df)
        print("Available figures for Mutual Information:", analysis_mi.figure_names)
    else:
        print("Error: exp.analysis is None. Cannot set options or run analysis.")

    # Try to display the figure
    try:
        if analysis_mi.figure_names:
            fig = analysis_mi.figure(0)
            import matplotlib.pyplot as plt
            plt.show()
        else:
            print("No figures available for Mutual Information.")
    except Exception as e:
        print(f"Error accessing figure for Mutual Information: {e}")
except Exception as e:
    print(f"Error during analysis for Mutual Information: {e}")

# Custom plotting based on original intent using QuantumAwesomeness
try:
    import matplotlib.pyplot as plt
    import numpy as np

    # Extract lengths from rb_data (using xval or index as fallback)
    data_entries = rb_data.data()
    lengths = [entry['metadata'].get('xval', i) for i, entry in enumerate(data_entries)]
    unique_lengths = sorted(list(set(lengths)))  # Unique lengths: [2, 4, 10]
    print("Derived unique lengths:", unique_lengths)

    # Initialize data structures
    num_qubits = len(exp.physical_qubits)  # Assuming 5 qubits from Q0-Q4
    ys = [[[] for _ in range(len(unique_lengths))] for _ in range(2)]  # 2 types x unique lengths
    yerrs = [[], []]

    # Compute mutual information using QuantumAwesomeness
    qa = QuantumAwesomeness(exp.backend.coupling_map)
    mmi = qa.mean_mutual_info(data_entries, [entry['metadata']['pairs'] for entry in data_entries])
    print("Mean mutual info:", mmi)

    # Aggregate mmi values by unique lengths, filtering nan
    for i, entry in enumerate(data_entries):
        length_idx = unique_lengths.index(entry['metadata'].get('xval', i))
        paired_val = mmi['paired'][i]
        single_val = mmi['single'][i]
        if not np.isnan(paired_val):
            ys[0][length_idx].append(paired_val)
        if not np.isnan(single_val):
            ys[1][length_idx].append(single_val)

    # Compute means and standard deviations, handling empty lists
    for p in range(2):
        for j in range(len(unique_lengths)):
            if ys[p][j]:
                yerrs[p].append(np.std(ys[p][j]))
                ys[p][j] = np.mean(ys[p][j])
            else:
                yerrs[p].append(0)  # Default to 0 error if no data
                ys[p][j] = 0       # Default to 0 mean if no data

    # Plot with error bars
    plt.errorbar(unique_lengths, ys[0], yerr=yerrs[0], fmt='o-', label='paired', capsize=5)
    plt.errorbar(unique_lengths, ys[1], yerr=yerrs[1], fmt='o-', label='singles', capsize=5)
    plt.yscale('log')
    plt.xlabel('Circuit Length (xval)')
    plt.ylabel('Mean Mutual Information')
    plt.title('Mutual Information vs Circuit Length')
    plt.legend()
    plt.show()

except Exception as e:
    print(f"Error during custom plotting: {e}")

In [ ]:
ep_results = analysis.analysis_results(dataframe=True)
print("Type of results:", type(ep_results))
print(ep_results.head())  # Show first few rows if it's a DataFrame
mi_results = analysis_mi.analysis_results(dataframe=True)
print(mi_results.head())

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
import pandas as pd 
InteractiveShell.instance().ast_node_interactivity = "all"


pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns


# Debug: Inspect rb_data contents
print("Experiment data contents:")
try:
    data_entries = rb_data.data()
    for i, data in enumerate(data_entries):
        print(f"Data entry {i}: {data}")
        # Check required metadata
        required_keys = ["pairs", "singles", "target", "coupling_map"]
        metadata = data.get("metadata", {})
        for key in required_keys:
            print(f"  Metadata '{key}' present: {key in metadata}")
except Exception as e:
    print(f"Error accessing rb_data.data(): {e}")

# Save rb_data to a file for inspection
try:
    with open("rb_data.pkl", "wb") as f:
        pickle.dump(rb_data, f)
    print("Saved rb_data to rb_data.pkl for further inspection")
except Exception as e:
    print(f"Error saving rb_data: {e}")

# Try analysis with "Effective Polarization"
try:
    exp.analysis.set_options(analyzed_quantity="Effective Polarization")
    print("Running analysis for Effective Polarization...")
    start_time = time.time()
    analysis = exp.analysis.run(rb_data)
    end_time = time.time()
    print(f"Analysis for Effective Polarization completed in {end_time - start_time:.2f} seconds")
    ep_results_df = analysis.analysis_results(dataframe=True)
    print("Analysis results for Effective Polarization:\n", ep_results_df)
    print("Available figures for Effective Polarization:", analysis.figure_names)

    # Try to display the figure
    try:
        fig = analysis.figure(0)
        plt.show()
    except Exception as e:
        print(f"Error accessing figure for Effective Polarization: {e}")

    # Extract results into DataFrame and plot
    import pandas as pd
    import matplotlib.pyplot as plt
    if not ep_results_df.empty:
        print("Debug: Available metrics in DataFrame:", ep_results_df['name'].unique())
        metrics = ['alpha', 'EPC', 'EI', 'chisq']
        values = []
        for metric in metrics:
            if (ep_results_df['name'] == metric).any():
                value = ep_results_df[ep_results_df['name'] == metric]['value'].iloc[0]
                if pd.isna(value) or value <= 0:
                    print(f"Warning: Invalid value for {metric}, skipping.")
                else:
                    values.append(float(value.n) if hasattr(value, 'n') else float(value))
            else:
                print(f"Metric {metric} not found in DataFrame.")
        
        if values:
            plt.figure(figsize=(10, 6))
            plt.bar([m for m, v in zip(metrics, values) if v is not None], values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'][:len(values)])
            plt.xlabel('Metrics')
            plt.ylabel('Value')
            plt.title('Effective Polarization Metrics')
            # plt.yscale('log')  # Comment out log scale for now, uncomment if needed
            for i, v in enumerate(values):
                plt.text(i, v, f'{v:.2e}' if v > 0 else str(v), ha='center', va='bottom')
            plt.show()
        else:
            print("No valid metrics found for plotting.")
    else:
        print("DataFrame is empty, no data to plot.")

except Exception as e:
    print(f"Error during analysis for Effective Polarization: {e}")

# Try analysis with "Mutual Information" as a fallback
try:
    exp.analysis.set_options(analyzed_quantity="Mutual Information")
    print("Running analysis for Mutual Information...")
    start_time = time.time()
    analysis_mi = exp.analysis.run(rb_data)
    end_time = time.time()
    print(f"Analysis for Mutual Information completed in {end_time - start_time:.2f} seconds")
    mi_results_df = analysis_mi.analysis_results(dataframe=True)
    print("Analysis results for Mutual Information:\n", mi_results_df)
    print("Available figures for Mutual Information:", analysis_mi.figure_names)

    # Try to display the figure
    try:
        fig = analysis_mi.figure(0)
        plt.show()
    except Exception as e:
        print(f"Error accessing figure for Mutual Information: {e}")

    # Extract results into DataFrame and plot
    if not mi_results_df.empty:
        print("Debug: Available metrics in DataFrame:", mi_results_df['name'].unique())
        metrics = ['alpha', 'EPC', 'EI', 'chisq']
        values = []
        for metric in metrics:
            if (mi_results_df['name'] == metric).any():
                value = mi_results_df[mi_results_df['name'] == metric]['value'].iloc[0]
                if pd.isna(value) or value <= 0:
                    print(f"Warning: Invalid value for {metric}, skipping.")
                else:
                    values.append(float(value.n) if hasattr(value, 'n') else float(value))
            else:
                print(f"Metric {metric} not found in DataFrame.")
        
        if values:
            plt.figure(figsize=(10, 6))
            plt.bar([m for m, v in zip(metrics, values) if v is not None], values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'][:len(values)])
            plt.xlabel('Metrics')
            plt.ylabel('Value')
            plt.title('Mutual Information Metrics')
            # plt.yscale('log')  # Comment out log scale for now, uncomment if needed
            for i, v in enumerate(values):
                plt.text(i, v, f'{v:.2e}' if v > 0 else str(v), ha='center', va='bottom')
            plt.show()
        else:
            print("No valid metrics found for plotting.")
    else:
        print("DataFrame is empty, no data to plot.")

except Exception as e:
    print(f"Error during analysis for Mutual Information: {e}")

# Fallback: Manual plotting if no figures are generated
try:
    if not analysis.figure_names and not analysis_mi.figure_names:
        print("No figures generated. Attempting manual plotting of results.")
        results = analysis.analysis_results() or analysis_mi.analysis_results()
        if results:
            lengths_data = []
            y_data = []
            for result in results:
                if hasattr(result, "value") and hasattr(result, "extra") and "length" in result.extra:
                    lengths_data.append(result.extra["length"])
                    y_data.append(result.value.n if hasattr(result.value, "n") else result.value)
        
            if lengths_data and y_data:
                plt.plot(lengths_data, y_data, 'o-', label="Analysis Results")
                plt.xlabel("Circuit Length")
                plt.ylabel("Value")
                plt.title("MirrorQA Analysis")
                plt.legend()
                plt.show()
            else:
                print("No valid data for manual plotting.")
        else:
            print("No analysis results available for plotting.")
except NameError:
    print("Analysis objects not defined due to earlier errors. Skipping manual plotting.")